# Game Week Points Prediction Notebook

This notebook trains an ML model to perform predictions on the number of points a player is predicted to achieve during the next gameweek. The data used in the project was pulled from the [Fantasy Premier League Repo](https://github.com/vaastav/Fantasy-Premier-League/tree/master).

## Load Data

First we will need to load the data used for the project as well as the libraries required for the project.


In [ ]:
import random

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import optuna

from optuna.samplers import TPESampler

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler, Normalizer
from sklearn.metrics import mean_squared_error, make_scorer

from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.svm import SVR

In [ ]:
data = pd.read_csv("data/cleaned_merged_seasons.csv")
data

In [ ]:
data_agg = pd.read_csv("data/cleaned_merged_seasons_team_aggregated.csv")
data_agg = pd.get_dummies(data_agg, columns=["position", "team_x", "opp_team_name"])
data_agg

In [ ]:
season_data = data_agg.groupby(
        ["season_x", "name"],
    ).agg(
        {
            "yellow_cards": "sum",
            "red_cards": "sum",
            "assists": "sum",
            "own_goals": "sum",
            "penalties_saved": "sum",
            "penalties_missed": "sum",
            "goals_scored": "sum",
            "goals_conceded": "sum",
            "total_points": "sum",
            "clean_sheets": "sum",
            "minutes": "sum",
            "ict_index": "mean",
            "influence": "mean",
        }
    ).reset_index().sort_values("season_x")
season_data["last_season"] = season_data.groupby("name").season_x.shift(-1)

season_data

In [ ]:
value_cols = ["goals_scored", 
              "assists", 
              "clean_sheets", 
              "yellow_cards", 
              "red_cards", 
              "minutes", 
              "own_goals", 
              "penalties_saved", 
              "penalties_missed", 
              "goals_conceded",
              "total_points"]

data_agg[[f"{c}_cum_prior" for c in value_cols]] = (
    data_agg.groupby("season_x")[value_cols]
      .cumsum()
      .shift(1)
      .fillna(0)
)

data_agg.rename(columns=lambda x: x.strip().replace("'", ""), inplace=True)
data_agg

In [ ]:
final_data = data_agg.merge(season_data, left_on=["season_x", "name"], right_on=["last_season", "name"], how="left")

In [ ]:
final_data = final_data.iloc[:, list(range(2)) + list(range(38, final_data.shape[1])) + list([26])]
final_data.drop(columns=["season_x_y", "last_season"], inplace=True)
final_data

## Train ML Model

Now we have the final dataframe and features, we can begin to train a model to predict the number of points achieved in the next game.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression


random.seed(445)

STORAGE = "sqlite:///db.sqlite3"

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(final_data.iloc[:, 2:89], final_data["total_points_x"] ,
                                   test_size=0.25, 
                                   shuffle=True)

### Linear Regression

First we'll train a simple linear regression as a baseline model

In [ ]:
regression_model = LinearRegression()
regression_model.fit(X_train.fillna(0), y_train)

y_preds = regression_model.predict(X_test.fillna(0))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, max_error, median_absolute_error, r2_score, explained_variance_score
import numpy as np


def regression_report(y_true, y_pred):
    
    error = y_true - y_pred
    percentil = [5,25,50,75,95]
    percentil_value = np.percentile(error, percentil)
    
    metrics = [
        ('mean absolute error', mean_absolute_error(y_true, y_pred)),
        ('median absolute error', median_absolute_error(y_true, y_pred)),
        ('mean squared error', mean_squared_error(y_true, y_pred)),
        ('max error', max_error(y_true, y_pred)),
        ('r2 score', r2_score(y_true, y_pred)),
        ('explained variance score', explained_variance_score(y_true, y_pred))
    ]
    
    print('Metrics for regression:')
    for metric_name, metric_value in metrics:
        print(f'{metric_name:>25s}: {metric_value: >20.3f}')
        
    print('\nPercentiles:')
    for p, pv in zip(percentil, percentil_value):
        print(f'{p: 25d}: {pv:>20.3f}')

In [ ]:
regression_report(y_test, y_preds)

In [ ]:
plt.hist(y_test - y_preds, bins=100)
plt.title("Histogram of Errors: Linear Regression")
plt.ylabel("Count of Predictions")
plt.xlabel("Point Prediction Error")

### LASSO Regression

Next, we'll train a LASSO regression model with hyper parameter tuning.

In [ ]:
X_train, X_validate, y_train, y_validate = train_test_split(X_train.fillna(0), 
                                                            y_train,
                                                            test_size=0.25, 
                                                            shuffle=True)


def ridge_objective(trial):
    model = Ridge(
        alpha=trial.suggest_float("alpha", 1e-3, 100, log=True),
        max_iter=trial.suggest_int("max_iter", 10, 1_000),
        tol=trial.suggest_float("tol", 0.00001, 0.9999, log=True),
        solver=trial.suggest_categorical("solver", ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga'])
    )
    model.fit(X_train, y_train)
    
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    scores = cross_val_score(
        pipe,
        X_validate,
        y_validate,
        cv=5,
        scoring=make_scorer(mean_squared_error, greater_is_better=False),
        n_jobs=1
    )

    return abs(scores.mean())


# ----------------------------
# Run optimization
# ----------------------------
sampler = TPESampler(n_startup_trials=50)
study = optuna.create_study(study_name="ridge_trial", direction="minimize", storage=STORAGE, sampler=sampler)
study.optimize(ridge_objective, n_trials=300, show_progress_bar=True)

print("Best score:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

### Ridge Regression

Next, we'll train a ridge regression model

In [ ]:
def ridge_objective(trial):
    model = Lasso(
            alpha=trial.suggest_float("alpha", 1e-3, 10, log=True),
            tol=trial.suggest_float("tol", 0.00001, 0.99999),
            max_iter=trial.suggest_int("max_iter", 10, 1000),
            selection=trial.suggest_categorical("selection", ['cyclic', 'random'])
        )
    
    model.fit(X_train, y_train)


    
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    scores = cross_val_score(
        pipe,
        X_validate,
        y_validate,
        cv=5,
        scoring=make_scorer(mean_squared_error, greater_is_better=False),
        n_jobs=1
    )

    return abs(scores.mean())


# ----------------------------
# Run optimization
# ----------------------------
sampler = TPESampler(n_startup_trials=50)
study = optuna.create_study(study_name="lasso_trial", direction="minimize", storage=STORAGE, sampler=sampler)
study.optimize(ridge_objective, n_trials=300, show_progress_bar=True)

print("Best score:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

### Random Forest Regression

Then we'll train a RF Regression model and hyperparameter tune using Optuna.

In [ ]:
def rf_objective(trial):
    model = RandomForestRegressor(n_estimators=trial.suggest_int("n_estimators", 100, 1000),
                                  max_depth=trial.suggest_int("max_depth", 3, 300),
                                  min_samples_split=trial.suggest_int("min_samples_split", 2, 1200),
                                  min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 1200),
                                  max_features=trial.suggest_int("max_features", 1, 800),
                                  min_impurity_decrease=trial.suggest_float("min_impurity", 0.0, 0.9999),
                                  random_state=42,
                                  n_jobs=1
                                 )
    
    model.fit(X_train, y_train)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    scores = cross_val_score(
        pipe,
        X_validate,
        y_validate,
        cv=5,
        scoring=make_scorer(mean_squared_error, greater_is_better=False),
        n_jobs=1
    )

    return abs(scores.mean())

sampler = TPESampler(n_startup_trials=50)
study = optuna.create_study(study_name="rf_trial", direction="minimize", storage=STORAGE, sampler=sampler)
study.optimize(rf_objective, n_trials=120, show_progress_bar=True)

print("Best score:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

### Extra Trees Regressor

Then we'll train up an extra trees regressor.

In [ ]:
def etr_objective(trial):
    model = ExtraTreesRegressor(
            n_estimators=trial.suggest_int("n_estimators", 100, 1200),
            max_depth=trial.suggest_int("max_depth", 3, 70),
            max_features=trial.suggest_float("max_features", 0.00001, 0.9999),
            max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 3, 800),
            min_samples_split=trial.suggest_int("min_samples_split", 3, 1000),
            min_samples_leaf=trial.suggest_float("min_samples_leaf", 0.00001, 0.9999),
            criterion=trial.suggest_categorical("criterion", ['squared_error', 'absolute_error', 'friedman_mse']),
            random_state=42,
            n_jobs=1
        )
    
    model.fit(X_train.fillna(0), y_train)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    scores = cross_val_score(
        pipe,
        X_validate,
        y_validate,
        cv=5,
        scoring=make_scorer(mean_squared_error, greater_is_better=False),
        n_jobs=1
    )

    return abs(scores.mean())


sampler = TPESampler(n_startup_trials=50)
study = optuna.create_study(study_name="etr_trial", direction="minimize", storage=STORAGE, sampler=sampler)
study.optimize(etr_objective, n_trials=300, show_progress_bar=True)

print("Best score:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

### Support Vector Regression

Finally, we will train up a SVR model.

In [ ]:
def svr_objective(trial):
    model = SVR(
            C=trial.suggest_float("C", 1e-2, 100, log=True),
            epsilon=trial.suggest_float("epsilon", 1e-3, 1.0, log=True),
            degree=trial.suggest_int("degree", 1, 100),
            gamma=trial.suggest_categorical("gamma", ['scale', 'auto']),
            coef0=trial.suggest_float("coef0", 0, 1, log=True),
            tol=trial.suggest_float("tol", 0,  0.9, log=True),
            kernel=trial.suggest_categorical("kernel", ['linear', 'poly', 'rbf', 'sigmoid', 'precomputed'])
        )

    model.fit(X_train, y_train)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    scores = cross_val_score(
        pipe,
        X_validate,
        y_validate,
        cv=5,
        scoring=make_scorer(mean_squared_error, greater_is_better=False),
        n_jobs=1
    )

    return abs(scores.mean())


sampler = TPESampler(n_startup_trials=50)
study = optuna.create_study(study_name="svr_trial", direction="minimize", storage=STORAGE, sampler=sampler)
study.optimize(svr_objective, n_trials=300, show_progress_bar=True)

print("Best score:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
#elif model_name == "lasso":
#        model = Lasso(
#            alpha=trial.suggest_float("alpha", 1e-3, 10, log=True),
#            max_iter=trial.suggest_int("max_iter", 10, 1000)
#        )
#        model.fit(X_train, y_train)
#
#    elif model_name == "random_forest":
#        model = RandomForestRegressor(
#            n_estimators=trial.suggest_int("n_estimators", 100, 500),
#            max_depth=trial.suggest_int("max_depth", 3, 30),
#            min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
#            random_state=42,
#            n_jobs=1
#        )
#        model.fit(X_train, y_train)
#    
#    elif model_name == "extra_trees":
#        model = ExtraTreesRegressor(
#            n_estimators=trial.suggest_int("n_estimators", 100, 500),
#            max_depth=trial.suggest_int("max_depth", 3, 30),
#            random_state=42,
#            n_jobs=1
#        )
#        model.fit(X_train, y_train)
#
#    else:  # SVR
#        model = SVR(
#            C=trial.suggest_float("C", 1e-2, 100, log=True),
#            epsilon=trial.suggest_float("epsilon", 1e-3, 1.0, log=True)
#        )
#        model.fit(X_train, y_train)